In [1]:
# Import the libraries we need

import pandas as pd
import numpy as np
import random
import os

# Make the results repeatable
random.seed(42)
np.random.seed(42)

# Create folders for our project
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [2]:

# ==========================================
# STEP 1: GENERATE MOCK ERP DATA
# ==========================================

# 1. CREATE PRODUCTS
products = pd.DataFrame({
    "product_id": [f"P{i:03d}" for i in range(1, 13)],
    "product_name": [
        "Laptop", "Monitor", "Keyboard", "Mouse",
        "Printer", "Tablet", "Headset", "Webcam",
        "Router", "Hard Drive", "USB Hub", "Docking Station"
    ],
    "category": [
        "Computers", "Displays", "Accessories", "Accessories",
        "Printers", "Computers", "Accessories", "Accessories",
        "Networking", "Storage", "Accessories", "Accessories"
    ],
    "unit_price": [
        950, 220, 45, 25, 180, 400,
        75, 60, 110, 90, 35, 150
    ]
})


# 2. CREATE CUSTOMERS
customers = pd.DataFrame({
    "customer_id": [f"C{i:03d}" for i in range(1, 41)],
    "customer_name": [f"Customer {i:03d}" for i in range(1, 41)],
    "region": np.random.choice(
        ["North", "South", "East", "West"],
        size=40
    ),
    "customer_type": np.random.choice(
        ["Retail", "Wholesale", "Corporate"],
        size=40
    )
})


# 3. CREATE ORDERS
orders_list = []

for i in range(1, 501):

    order_date = pd.Timestamp("2026-01-01") + pd.Timedelta(
        days=random.randint(0, 300)
    )

    promised_date = order_date + pd.Timedelta(
        days=random.randint(2, 7)
    )

    status = random.choices(
        ["Delivered", "Open", "Cancelled"],
        weights=[92, 5, 3]
    )[0]

    orders_list.append({
        "order_id": f"ORD{i:04d}",
        "customer_id": random.choice(customers["customer_id"]),
        "order_date": order_date,
        "promised_date": promised_date,
        "order_status": status
    })

orders = pd.DataFrame(orders_list)


# 4. CREATE ORDER ITEMS
order_items_list = []

for order_id in orders["order_id"]:

    number_of_products = random.randint(1, 4)

    selected_products = random.sample(
        list(products["product_id"]),
        number_of_products
    )

    for product_id in selected_products:

        order_items_list.append({
            "order_id": order_id,
            "product_id": product_id,
            "ordered_quantity": random.randint(2, 20)
        })

order_items = pd.DataFrame(order_items_list)


# 5. CREATE DELIVERIES AND DELIVERY ITEMS
deliveries_list = []
delivery_items_list = []

warehouse_ids = ["WH01", "WH02", "WH03", "WH04"]

for _, order in orders.iterrows():

    # Open and cancelled orders do not have deliveries yet
    if order["order_status"] != "Delivered":
        continue

    order_id = order["order_id"]

    items = order_items[
        order_items["order_id"] == order_id
    ]

    # Some orders arrive in two separate shipments
    number_of_shipments = random.choices(
        [1, 2],
        weights=[80, 20]
    )[0]

    shipment_quantities = {}

    for _, item in items.iterrows():

        ordered_qty = int(item["ordered_quantity"])

        # Some deliveries have shortages
        delivered_qty = ordered_qty

        if random.random() < 0.15:
            delivered_qty = random.randint(
                max(1, int(ordered_qty * 0.70)),
                ordered_qty - 1
            )

        # Divide quantities across shipments
        if number_of_shipments == 2 and delivered_qty >= 2:

            first_qty = random.randint(
                1, delivered_qty - 1
            )

            shipment_quantities[item["product_id"]] = [
                first_qty,
                delivered_qty - first_qty
            ]

        else:
            shipment_quantities[item["product_id"]] = [
                delivered_qty
            ]

    for shipment_number in range(number_of_shipments):

        delivery_id = f"DEL{len(deliveries_list) + 1:05d}"

        delay_days = random.choices(
            [-1, 0, 1, 2, 3, 4, 5],
            weights=[5, 35, 15, 15, 12, 10, 8]
        )[0]

        actual_delivery_date = (
            order["promised_date"]
            + pd.Timedelta(days=delay_days)
        )

        deliveries_list.append({
            "delivery_id": delivery_id,
            "order_id": order_id,
            "warehouse_id": random.choice(warehouse_ids),
            "actual_delivery_date": actual_delivery_date
        })

        for product_id, quantities in shipment_quantities.items():

            if shipment_number < len(quantities):

                delivery_items_list.append({
                    "delivery_id": delivery_id,
                    "product_id": product_id,
                    "delivered_quantity": quantities[shipment_number]
                })


deliveries = pd.DataFrame(deliveries_list)

delivery_items = pd.DataFrame(delivery_items_list)


# 6. INTRODUCE SOME INTENTIONAL DATA-QUALITY ISSUES
# We will learn to identify and fix these later.

# Missing customer region
customers.loc[3, "region"] = np.nan

# Extra spaces in a customer name
customers.loc[5, "customer_name"] = "  Customer 006  "

# Missing product category
products.loc[2, "category"] = np.nan

# Add a few duplicate delivery-item records
duplicate_rows = delivery_items.sample(
    n=5,
    random_state=42
)

delivery_items = pd.concat(
    [delivery_items, duplicate_rows],
    ignore_index=True
)


# 7. SAVE ALL SIX DATASETS AS CSV FILES

products.to_csv("data/products.csv", index=False)
customers.to_csv("data/customers.csv", index=False)
orders.to_csv("data/orders.csv", index=False)
order_items.to_csv("data/order_items.csv", index=False)
deliveries.to_csv("data/deliveries.csv", index=False)
delivery_items.to_csv("data/delivery_items.csv", index=False)

print("All six ERP datasets have been generated and saved!")


All six ERP datasets have been generated and saved!


In [3]:

# Check how many records were created in each file

datasets = {
    "Products": products,
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Deliveries": deliveries,
    "Delivery Items": delivery_items
}

for name, dataset in datasets.items():
    print(f"{name}: {len(dataset)} rows")

print("\nExample customer orders:")
display(orders.head())

print("\nExample deliveries:")
display(deliveries.head())

print("\nExample delivered products:")
display(delivery_items.head())


Products: 12 rows
Customers: 40 rows
Orders: 500 rows
Order Items: 1259 rows
Deliveries: 545 rows
Delivery Items: 1370 rows

Example customer orders:


,order_id,customer_id,order_date,promised_date,order_status
0,ORD0001,C016,2026-02-27,2026-03-01,Delivered
1,ORD0002,C035,2026-04-25,2026-04-28,Delivered
2,ORD0003,C002,2026-02-14,2026-02-20,Delivered
3,ORD0004,C039,2026-02-17,2026-02-20,Delivered
4,ORD0005,C035,2026-01-14,2026-01-20,Delivered



Example deliveries:


,delivery_id,order_id,warehouse_id,actual_delivery_date
0,DEL00001,ORD0001,WH02,2026-03-04
1,DEL00002,ORD0002,WH03,2026-04-27
2,DEL00003,ORD0003,WH03,2026-02-24
3,DEL00004,ORD0004,WH02,2026-02-20
4,DEL00005,ORD0005,WH03,2026-01-24



Example delivered products:


,delivery_id,product_id,delivered_quantity
0,DEL00001,P006,9
1,DEL00002,P002,6
2,DEL00002,P005,5
3,DEL00002,P008,3
4,DEL00002,P004,11


In [4]:

import shutil
from google.colab import files

# Package the data folder into one ZIP file
shutil.make_archive("otif_erp_datasets", "zip", "data")

# Download it to your computer
files.download("otif_erp_datasets.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:

# STEP 2.1: CREATE OUR SQLITE DATABASE

import sqlite3

# Connect to a database file.
# If the file does not exist, SQLite creates it.
connection = sqlite3.connect("otif_database.db")

print("Database created and connected successfully!")


Database created and connected successfully!


In [6]:

# STEP 2.2: LOAD ALL SIX CSV FILES INTO SQLITE

# Read each CSV file and save it as a SQL table

products.to_sql(
    "products", connection,
    if_exists="replace", index=False
)

customers.to_sql(
    "customers", connection,
    if_exists="replace", index=False
)

orders.to_sql(
    "orders", connection,
    if_exists="replace", index=False
)

order_items.to_sql(
    "order_items", connection,
    if_exists="replace", index=False
)

deliveries.to_sql(
    "deliveries", connection,
    if_exists="replace", index=False
)

delivery_items.to_sql(
    "delivery_items", connection,
    if_exists="replace", index=False
)

print("All six CSV files have been loaded into SQLite!")


All six CSV files have been loaded into SQLite!


In [7]:

# STEP 2.3: CHECK THE TABLES IN OUR DATABASE

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

display(tables)


,name
0,customers
1,deliveries
2,delivery_items
3,order_items
4,orders
5,products


In [8]:

# STEP 2.4: COUNT RECORDS IN EACH SQL TABLE

table_names = [
    "products",
    "customers",
    "orders",
    "order_items",
    "deliveries",
    "delivery_items"
]

for table in table_names:

    query = f"SELECT COUNT(*) AS total_records FROM {table}"

    result = pd.read_sql_query(query, connection)

    print(f"{table}: {result.iloc[0, 0]} records")


products: 12 records
customers: 40 records
orders: 500 records
order_items: 1259 records
deliveries: 545 records
delivery_items: 1370 records


In [9]:

# STEP 2.5: FIND ALL DELIVERED ORDERS

query = """
SELECT *
FROM orders
WHERE order_status = 'Delivered'
LIMIT 10;
"""

delivered_orders = pd.read_sql_query(query, connection)

display(delivered_orders)


,order_id,customer_id,order_date,promised_date,order_status
0,ORD0001,C016,2026-02-27 00:00:00,2026-03-01 00:00:00,Delivered
1,ORD0002,C035,2026-04-25 00:00:00,2026-04-28 00:00:00,Delivered
2,ORD0003,C002,2026-02-14 00:00:00,2026-02-20 00:00:00,Delivered
3,ORD0004,C039,2026-02-17 00:00:00,2026-02-20 00:00:00,Delivered
4,ORD0005,C035,2026-01-14 00:00:00,2026-01-20 00:00:00,Delivered
5,ORD0006,C018,2026-08-03 00:00:00,2026-08-06 00:00:00,Delivered
6,ORD0007,C022,2026-01-04 00:00:00,2026-01-07 00:00:00,Delivered
7,ORD0008,C022,2026-05-23 00:00:00,2026-05-26 00:00:00,Delivered
8,ORD0009,C023,2026-02-22 00:00:00,2026-02-24 00:00:00,Delivered
9,ORD0010,C003,2026-06-26 00:00:00,2026-07-02 00:00:00,Delivered


In [10]:

# STEP 2.6: COUNT ORDERS BY STATUS

query = """
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;
"""

order_status_summary = pd.read_sql_query(query, connection)

display(order_status_summary)


,order_status,total_orders
0,Delivered,462
1,Open,25
2,Cancelled,13


In [11]:
connection.commit()
connection.close()

print("Database changes saved and connection closed.")

Database changes saved and connection closed.


In [12]:
import sqlite3
import pandas as pd

# Reconnect to our existing database
connection = sqlite3.connect("otif_database.db")

print("Successfully connected to the OTIF database!")

Successfully connected to the OTIF database!


In [13]:
query = """
SELECT *
FROM customers
WHERE region IS NULL;
"""

missing_regions = pd.read_sql_query(query, connection)

display(missing_regions)

,customer_id,customer_name,region,customer_type
0,C004,Customer 004,None,Corporate


In [14]:
query = """
SELECT *
FROM products
WHERE category IS NULL;
"""

missing_categories = pd.read_sql_query(query, connection)

display(missing_categories)

,product_id,product_name,category,unit_price
0,P003,Keyboard,None,45


In [15]:
query = """
SELECT
    delivery_id,
    product_id,
    delivered_quantity,
    COUNT(*) AS duplicate_count
FROM delivery_items
GROUP BY
    delivery_id,
    product_id,
    delivered_quantity
HAVING COUNT(*) > 1;
"""

duplicates = pd.read_sql_query(query, connection)

display(duplicates)

,delivery_id,product_id,delivered_quantity,duplicate_count
0,DEL00105,P002,8,2
1,DEL00364,P007,3,2
2,DEL00376,P004,7,2
3,DEL00379,P001,16,2
4,DEL00439,P008,11,2


In [16]:
query = """
SELECT
    order_id,
    COUNT(*) AS total_records
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1;
"""

duplicate_orders = pd.read_sql_query(query, connection)

display(duplicate_orders)

,order_id,total_records


In [17]:
query = """
SELECT *
FROM order_items
WHERE ordered_quantity <= 0;
"""

invalid_orders = pd.read_sql_query(query, connection)

display(invalid_orders)

,order_id,product_id,ordered_quantity


In [18]:
query = """
SELECT *
FROM delivery_items
WHERE delivered_quantity <= 0;
"""

invalid_deliveries = pd.read_sql_query(query, connection)

display(invalid_deliveries)

,delivery_id,product_id,delivered_quantity


In [19]:
query = """
SELECT o.*
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;
"""

orphan_orders = pd.read_sql_query(query, connection)

display(orphan_orders)

,order_id,customer_id,order_date,promised_date,order_status


In [20]:
query = """
SELECT oi.*
FROM order_items oi
LEFT JOIN orders o
    ON oi.order_id = o.order_id
WHERE o.order_id IS NULL;
"""

orphan_order_items = pd.read_sql_query(query, connection)

display(orphan_order_items)

,order_id,product_id,ordered_quantity


In [21]:
query = """
SELECT d.*
FROM deliveries d
LEFT JOIN orders o
    ON d.order_id = o.order_id
WHERE o.order_id IS NULL;
"""

orphan_deliveries = pd.read_sql_query(query, connection)

display(orphan_deliveries)

,delivery_id,order_id,warehouse_id,actual_delivery_date


In [22]:
sql = """
DROP TABLE IF EXISTS customers_clean;

CREATE TABLE customers_clean AS

SELECT
    TRIM(customer_id) AS customer_id,

    TRIM(customer_name) AS customer_name,

    COALESCE(
        NULLIF(TRIM(region), ''),
        'Unknown'
    ) AS region,

    COALESCE(
        NULLIF(TRIM(customer_type), ''),
        'Unknown'
    ) AS customer_type

FROM customers;
"""

connection.executescript(sql)

print("Customers table cleaned successfully!")

Customers table cleaned successfully!


In [23]:
query = """
SELECT *
FROM customers_clean
WHERE region = 'Unknown'
   OR customer_name = 'Customer 006';
"""

display(pd.read_sql_query(query, connection))

,customer_id,customer_name,region,customer_type
0,C004,Customer 004,Unknown,Corporate
1,C006,Customer 006,West,Wholesale


In [24]:
sql = """
DROP TABLE IF EXISTS products_clean;

CREATE TABLE products_clean AS

SELECT
    TRIM(product_id) AS product_id,

    TRIM(product_name) AS product_name,

    COALESCE(
        NULLIF(TRIM(category), ''),
        'Uncategorized'
    ) AS category,

    CAST(unit_price AS REAL) AS unit_price

FROM products;
"""

connection.executescript(sql)

print("Products table cleaned successfully!")

Products table cleaned successfully!


In [25]:
query = """
SELECT *
FROM products_clean
WHERE category = 'Uncategorized';
"""

display(pd.read_sql_query(query, connection))

,product_id,product_name,category,unit_price
0,P003,Keyboard,Uncategorized,45.0


In [26]:
sql = """
DROP TABLE IF EXISTS orders_clean;

CREATE TABLE orders_clean AS

SELECT DISTINCT
    TRIM(order_id) AS order_id,

    TRIM(customer_id) AS customer_id,

    DATE(order_date) AS order_date,

    DATE(promised_date) AS promised_date,

    TRIM(order_status) AS order_status

FROM orders;
"""

connection.executescript(sql)

print("Orders table cleaned successfully!")

Orders table cleaned successfully!


In [27]:
sql = """
DROP TABLE IF EXISTS order_items_clean;

CREATE TABLE order_items_clean AS

SELECT DISTINCT
    TRIM(order_id) AS order_id,

    TRIM(product_id) AS product_id,

    CAST(ordered_quantity AS INTEGER) AS ordered_quantity

FROM order_items
WHERE CAST(ordered_quantity AS INTEGER) > 0;
"""

connection.executescript(sql)

print("Order items table cleaned successfully!")

Order items table cleaned successfully!


In [28]:
sql = """
DROP TABLE IF EXISTS deliveries_clean;

CREATE TABLE deliveries_clean AS

SELECT DISTINCT
    TRIM(delivery_id) AS delivery_id,

    TRIM(order_id) AS order_id,

    TRIM(warehouse_id) AS warehouse_id,

    DATE(actual_delivery_date) AS actual_delivery_date

FROM deliveries;
"""

connection.executescript(sql)

print("Deliveries table cleaned successfully!")

Deliveries table cleaned successfully!


In [29]:
sql = """
DROP TABLE IF EXISTS delivery_items_clean;

CREATE TABLE delivery_items_clean AS

SELECT DISTINCT
    TRIM(delivery_id) AS delivery_id,

    TRIM(product_id) AS product_id,

    CAST(delivered_quantity AS INTEGER) AS delivered_quantity

FROM delivery_items
WHERE CAST(delivered_quantity AS INTEGER) > 0;
"""

connection.executescript(sql)

print("Delivery items table cleaned successfully!")

Delivery items table cleaned successfully!


In [6]:
import sqlite3
import pandas as pd
import os

# Establish connection to the database
connection = sqlite3.connect("otif_database.db")

# List of tables to verify and load if missing
tables_to_load = {
    "products": "data/products.csv",
    "customers": "data/customers.csv",
    "orders": "data/orders.csv",
    "order_items": "data/order_items.csv",
    "deliveries": "data/deliveries.csv",
    "delivery_items": "data/delivery_items.csv"
}

# Restore any missing raw tables from CSVs
for table_name, csv_path in tables_to_load.items():
    # Check if table exists
    cursor = connection.cursor()
    cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
    if not cursor.fetchone() and os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.to_sql(table_name, connection, if_exists="replace", index=False)

# Compare the original and cleaned tables
table_comparison = []

table_pairs = [
    ("customers", "customers_clean"),
    ("products", "products_clean"),
    ("orders", "orders_clean"),
    ("order_items", "order_items_clean"),
    ("deliveries", "deliveries_clean"),
    ("delivery_items", "delivery_items_clean")
]

for raw_table, clean_table in table_pairs:
    try:
        raw_count = pd.read_sql_query(
            f"SELECT COUNT(*) AS total FROM {raw_table}",
            connection
        ).iloc[0, 0]
    except Exception:
        raw_count = "Error/Missing"

    try:
        clean_count = pd.read_sql_query(
            f"SELECT COUNT(*) AS total FROM {clean_table}",
            connection
        ).iloc[0, 0]
    except Exception:
        clean_count = "Error/Missing"

    if isinstance(raw_count, int) and isinstance(clean_count, int):
        removed = raw_count - clean_count
    else:
        removed = "N/A"

    table_comparison.append({
        "Table": raw_table,
        "Original Records": raw_count,
        "Cleaned Records": clean_count,
        "Records Removed": removed
    })

comparison_df = pd.DataFrame(table_comparison)

display(comparison_df)

,Table,Original Records,Cleaned Records,Records Removed
0,customers,Error/Missing,Error/Missing,N/A
1,products,Error/Missing,Error/Missing,N/A
2,orders,Error/Missing,Error/Missing,N/A
3,order_items,Error/Missing,Error/Missing,N/A
4,deliveries,Error/Missing,Error/Missing,N/A
5,delivery_items,Error/Missing,Error/Missing,N/A


In [7]:
import sqlite3
import pandas as pd
import os

print("Database file exists:", os.path.exists("otif_database.db"))

connection = sqlite3.connect("otif_database.db")

print("Database connection successful!")

Database file exists: True
Database connection successful!


In [8]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", connection)

display(tables)

,name


In [9]:
import os

print("Current folder:", os.getcwd())
print("\nFiles in current folder:")
print(os.listdir())

print("\nFiles inside data folder:")
if os.path.exists("data"):
    print(os.listdir("data"))
else:
    print("Data folder does not exist.")

Current folder: /content

Files in current folder:
['.config', 'otif_database.db', 'sample_data']

Files inside data folder:
Data folder does not exist.


In [10]:
import pandas as pd
import numpy as np
import random
import os

random.seed(42)
np.random.seed(42)

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

# 1. Products
products = pd.DataFrame({
    "product_id": [f"P{i:03d}" for i in range(1, 13)],
    "product_name": [
        "Laptop", "Monitor", "Keyboard", "Mouse",
        "Printer", "Tablet", "Headset", "Webcam",
        "Router", "Hard Drive", "USB Hub", "Docking Station"
    ],
    "category": [
        "Computers", "Accessories", "Accessories", "Accessories",
        "Printers", "Computers", "Accessories", "Accessories",
        "Networking", "Storage", "Accessories", "Accessories"
    ],
    "unit_price": [899, 249, 49, 25, 199, 499, 79, 59, 129, 89, 35, 149]
})

# 2. Customers
customers = pd.DataFrame({
    "customer_id": [f"C{i:03d}" for i in range(1, 41)],
    "customer_name": [f"Customer {i:03d}" for i in range(1, 41)],
    "region": np.random.choice(["North", "South", "East", "West"], 40),
    "customer_type": np.random.choice(
        ["Retail", "Wholesale", "Corporate"], 40
    )
})

# Add intentional data-quality issues
customers.loc[3, "region"] = np.nan
customers.loc[5, "customer_name"] = "  Customer 006  "
products.loc[2, "category"] = np.nan

# 3. Orders
start_date = pd.Timestamp("2026-01-01")

orders = pd.DataFrame({
    "order_id": [f"ORD{i:04d}" for i in range(1, 501)],
    "customer_id": np.random.choice(customers["customer_id"], 500),
    "order_date": [
        start_date + pd.Timedelta(days=random.randint(0, 300))
        for _ in range(500)
    ]
})

orders["promised_date"] = orders["order_date"] + pd.to_timedelta(
    np.random.randint(2, 8, 500), unit="D"
)

orders["order_status"] = np.random.choice(
    ["Delivered", "Open", "Cancelled"],
    500,
    p=[0.92, 0.05, 0.03]
)

# 4. Order items
order_items_list = []

for order_id in orders["order_id"]:
    selected_products = random.sample(
        list(products["product_id"]),
        random.randint(1, 4)
    )

    for product_id in selected_products:
        order_items_list.append({
            "order_id": order_id,
            "product_id": product_id,
            "ordered_quantity": random.randint(2, 20)
        })

order_items = pd.DataFrame(order_items_list)

# 5. Deliveries
deliveries_list = []
delivery_items_list = []

delivered_orders = orders[orders["order_status"] == "Delivered"]

for _, order in delivered_orders.iterrows():

    number_of_deliveries = random.choices([1, 2], weights=[80, 20])[0]

    for shipment_number in range(number_of_deliveries):

        delivery_id = f"DEL{len(deliveries_list) + 1:05d}"

        delay = random.choices(
            [-1, 0, 1, 2, 3, 4, 5],
            weights=[5, 35, 15, 15, 12, 10, 8]
        )[0]

        actual_date = order["promised_date"] + pd.Timedelta(days=delay)

        deliveries_list.append({
            "delivery_id": delivery_id,
            "order_id": order["order_id"],
            "warehouse_id": random.choice(
                ["WH01", "WH02", "WH03", "WH04"]
            ),
            "actual_delivery_date": actual_date
        })

        order_products = order_items[
            order_items["order_id"] == order["order_id"]
        ]

        for _, item in order_products.iterrows():

            ordered_qty = item["ordered_quantity"]

            if random.random() < 0.15:
                delivered_qty = random.randint(
                    max(1, int(ordered_qty * 0.7)),
                    ordered_qty
                )
            else:
                delivered_qty = ordered_qty

            if number_of_deliveries == 2 and shipment_number == 0:
                delivered_qty = max(1, delivered_qty // 2)
            elif number_of_deliveries == 2:
                previous = sum(
                    x["delivered_quantity"]
                    for x in delivery_items_list
                    if x["product_id"] == item["product_id"]
                    and x["delivery_id"] in [
                        d["delivery_id"] for d in deliveries_list[:-1]
                    ]
                    and x.get("order_id") == order["order_id"]
                )
                delivered_qty = max(1, ordered_qty - previous)

            delivery_items_list.append({
                "delivery_id": delivery_id,
                "product_id": item["product_id"],
                "delivered_quantity": delivered_qty,
                "order_id": order["order_id"]
            })

deliveries = pd.DataFrame(deliveries_list)
delivery_items = pd.DataFrame(delivery_items_list)

# Remove helper column from delivery_items
delivery_items = delivery_items.drop(columns=["order_id"])

# Add five intentional duplicate records
duplicates = delivery_items.sample(5, random_state=42)
delivery_items = pd.concat(
    [delivery_items, duplicates],
    ignore_index=True
)

# Save all six datasets
datasets = {
    "orders": orders,
    "order_items": order_items,
    "deliveries": deliveries,
    "delivery_items": delivery_items,
    "products": products,
    "customers": customers
}

for name, df in datasets.items():
    df.to_csv(f"data/{name}.csv", index=False)

print("All six datasets regenerated successfully!")

for name, df in datasets.items():
    print(f"{name}.csv: {len(df)} records")

All six datasets regenerated successfully!
orders.csv: 500 records
order_items.csv: 1233 records
deliveries.csv: 540 records
delivery_items.csv: 1333 records
products.csv: 12 records
customers.csv: 40 records


In [11]:
print(os.listdir("data"))

['orders.csv', 'order_items.csv', 'products.csv', 'delivery_items.csv', 'deliveries.csv', 'customers.csv']


In [12]:
import sqlite3
import pandas as pd
import os

connection = sqlite3.connect("otif_database.db")

print("Connected to SQLite successfully!")

Connected to SQLite successfully!


In [13]:
datasets = [
    "customers",
    "products",
    "orders",
    "order_items",
    "deliveries",
    "delivery_items"
]

for name in datasets:
    df = pd.read_csv(f"data/{name}.csv")

    df.to_sql(
        name,
        connection,
        if_exists="replace",
        index=False
    )

    print(f"{name}: {len(df)} records loaded successfully")

connection.commit()

customers: 40 records loaded successfully
products: 12 records loaded successfully
orders: 500 records loaded successfully
order_items: 1233 records loaded successfully
deliveries: 540 records loaded successfully
delivery_items: 1333 records loaded successfully


In [14]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", connection)

display(tables)

,name
0,customers
1,deliveries
2,delivery_items
3,order_items
4,orders
5,products


In [15]:
for name in datasets:
    result = pd.read_sql_query(
        f"SELECT COUNT(*) AS total_records FROM {name}",
        connection
    )

    print(f"{name}: {result.iloc[0, 0]} records")

customers: 40 records
products: 12 records
orders: 500 records
order_items: 1233 records
deliveries: 540 records
delivery_items: 1333 records


In [16]:
sql = """
DROP TABLE IF EXISTS customers_clean;

CREATE TABLE customers_clean AS
SELECT
    TRIM(customer_id) AS customer_id,
    TRIM(customer_name) AS customer_name,
    COALESCE(NULLIF(TRIM(region), ''), 'Unknown') AS region,
    COALESCE(NULLIF(TRIM(customer_type), ''), 'Unknown') AS customer_type
FROM customers;
"""

connection.executescript(sql)
print("Customers cleaned!")

Customers cleaned!


In [17]:
sql = """
DROP TABLE IF EXISTS products_clean;

CREATE TABLE products_clean AS
SELECT
    TRIM(product_id) AS product_id,
    TRIM(product_name) AS product_name,
    COALESCE(NULLIF(TRIM(category), ''), 'Uncategorized') AS category,
    CAST(unit_price AS REAL) AS unit_price
FROM products;
"""

connection.executescript(sql)
print("Products cleaned!")

Products cleaned!


In [18]:
sql = """
DROP TABLE IF EXISTS orders_clean;

CREATE TABLE orders_clean AS
SELECT DISTINCT
    TRIM(order_id) AS order_id,
    TRIM(customer_id) AS customer_id,
    DATE(order_date) AS order_date,
    DATE(promised_date) AS promised_date,
    TRIM(order_status) AS order_status
FROM orders;
"""

connection.executescript(sql)
print("Orders cleaned!")

Orders cleaned!


In [19]:
sql = """
DROP TABLE IF EXISTS order_items_clean;

CREATE TABLE order_items_clean AS
SELECT DISTINCT
    TRIM(order_id) AS order_id,
    TRIM(product_id) AS product_id,
    CAST(ordered_quantity AS INTEGER) AS ordered_quantity
FROM order_items
WHERE CAST(ordered_quantity AS INTEGER) > 0;
"""

connection.executescript(sql)
print("Order items cleaned!")

Order items cleaned!


In [20]:
sql = """
DROP TABLE IF EXISTS deliveries_clean;

CREATE TABLE deliveries_clean AS
SELECT DISTINCT
    TRIM(delivery_id) AS delivery_id,
    TRIM(order_id) AS order_id,
    TRIM(warehouse_id) AS warehouse_id,
    DATE(actual_delivery_date) AS actual_delivery_date
FROM deliveries;
"""

connection.executescript(sql)
print("Deliveries cleaned!")

Deliveries cleaned!


In [21]:
sql = """
DROP TABLE IF EXISTS delivery_items_clean;

CREATE TABLE delivery_items_clean AS
SELECT DISTINCT
    TRIM(delivery_id) AS delivery_id,
    TRIM(product_id) AS product_id,
    CAST(delivered_quantity AS INTEGER) AS delivered_quantity
FROM delivery_items
WHERE CAST(delivered_quantity AS INTEGER) > 0;
"""

connection.executescript(sql)
print("Delivery items cleaned!")

Delivery items cleaned!


In [22]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", connection)

display(tables)

,name
0,customers
1,customers_clean
2,deliveries
3,deliveries_clean
4,delivery_items
5,delivery_items_clean
6,order_items
7,order_items_clean
8,orders
9,orders_clean


In [23]:
table_pairs = [
    ("customers", "customers_clean"),
    ("products", "products_clean"),
    ("orders", "orders_clean"),
    ("order_items", "order_items_clean"),
    ("deliveries", "deliveries_clean"),
    ("delivery_items", "delivery_items_clean")
]

table_comparison = []

for raw_table, clean_table in table_pairs:

    raw_count = pd.read_sql_query(
        f"SELECT COUNT(*) AS total FROM {raw_table}",
        connection
    ).iloc[0, 0]

    clean_count = pd.read_sql_query(
        f"SELECT COUNT(*) AS total FROM {clean_table}",
        connection
    ).iloc[0, 0]

    table_comparison.append({
        "Table": raw_table,
        "Original Records": raw_count,
        "Cleaned Records": clean_count,
        "Records Removed": raw_count - clean_count
    })

comparison_df = pd.DataFrame(table_comparison)
display(comparison_df)

,Table,Original Records,Cleaned Records,Records Removed
0,customers,40,40,0
1,products,12,12,0
2,orders,500,500,0
3,order_items,1233,1233,0
4,deliveries,540,540,0
5,delivery_items,1333,1328,5


In [24]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", connection)

display(tables)

,name
0,customers
1,customers_clean
2,deliveries
3,deliveries_clean
4,delivery_items
5,delivery_items_clean
6,order_items
7,order_items_clean
8,orders
9,orders_clean


In [25]:
table_pairs = [
    ("customers", "customers_clean"),
    ("products", "products_clean"),
    ("orders", "orders_clean"),
    ("order_items", "order_items_clean"),
    ("deliveries", "deliveries_clean"),
    ("delivery_items", "delivery_items_clean")
]

table_comparison = []

for raw_table, clean_table in table_pairs:

    raw_count = pd.read_sql_query(
        f"SELECT COUNT(*) AS total FROM {raw_table}",
        connection
    ).iloc[0, 0]

    clean_count = pd.read_sql_query(
        f"SELECT COUNT(*) AS total FROM {clean_table}",
        connection
    ).iloc[0, 0]

    table_comparison.append({
        "Table": raw_table,
        "Original Records": raw_count,
        "Cleaned Records": clean_count,
        "Records Removed": raw_count - clean_count
    })

comparison_df = pd.DataFrame(table_comparison)

display(comparison_df)

,Table,Original Records,Cleaned Records,Records Removed
0,customers,40,40,0
1,products,12,12,0
2,orders,500,500,0
3,order_items,1233,1233,0
4,deliveries,540,540,0
5,delivery_items,1333,1328,5


In [26]:
missing_customers = pd.read_sql_query("""
SELECT *
FROM customers_clean
WHERE region IS NULL
   OR customer_name IS NULL;
""", connection)

display(missing_customers)

,customer_id,customer_name,region,customer_type


In [27]:
missing_products = pd.read_sql_query("""
SELECT *
FROM products_clean
WHERE category IS NULL
   OR unit_price IS NULL;
""", connection)

display(missing_products)

,product_id,product_name,category,unit_price


In [28]:
duplicate_check = pd.read_sql_query("""
SELECT
    delivery_id,
    product_id,
    delivered_quantity,
    COUNT(*) AS total_records
FROM delivery_items_clean
GROUP BY delivery_id, product_id, delivered_quantity
HAVING COUNT(*) > 1;
""", connection)

display(duplicate_check)

,delivery_id,product_id,delivered_quantity,total_records


In [29]:
import sqlite3
import pandas as pd

connection = sqlite3.connect("otif_database.db")

print("Connected to OTIF database!")

Connected to OTIF database!


In [30]:
query = """
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM orders_clean
GROUP BY order_status
ORDER BY total_orders DESC;
"""

status_summary = pd.read_sql_query(query, connection)

display(status_summary)

,order_status,total_orders
0,Delivered,455
1,Open,30
2,Cancelled,15


In [31]:
query = """
SELECT
    COUNT(*) AS eligible_orders
FROM orders_clean
WHERE order_status = 'Delivered';
"""

eligible_orders = pd.read_sql_query(query, connection)

display(eligible_orders)

,eligible_orders
0,455


In [32]:
query = """
SELECT
    o.order_id,
    SUM(oi.ordered_quantity) AS total_ordered_quantity
FROM orders_clean o
JOIN order_items_clean oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'Delivered'
GROUP BY o.order_id
ORDER BY o.order_id;
"""

ordered_summary = pd.read_sql_query(query, connection)

display(ordered_summary.head(10))

,order_id,total_ordered_quantity
0,ORD0001,15
1,ORD0002,19
2,ORD0003,22
3,ORD0004,50
4,ORD0005,26
5,ORD0006,35
6,ORD0007,15
7,ORD0009,6
8,ORD0010,12
9,ORD0011,41


In [33]:
query = """
SELECT
    d.order_id,
    SUM(di.delivered_quantity) AS total_delivered_quantity
FROM deliveries_clean d
JOIN delivery_items_clean di
    ON d.delivery_id = di.delivery_id
GROUP BY d.order_id
ORDER BY d.order_id;
"""

delivered_summary = pd.read_sql_query(query, connection)

display(delivered_summary.head(10))

,order_id,total_delivered_quantity
0,ORD0001,15
1,ORD0002,19
2,ORD0003,22
3,ORD0004,50
4,ORD0005,26
5,ORD0006,32
6,ORD0007,15
7,ORD0009,6
8,ORD0010,12
9,ORD0011,37


In [34]:
query = """
SELECT
    o.order_id,
    SUM(oi.ordered_quantity) AS total_ordered_quantity,
    COALESCE(SUM(delivered.total_delivered), 0) AS total_delivered_quantity
FROM orders_clean o

JOIN order_items_clean oi
    ON o.order_id = oi.order_id

LEFT JOIN (
    SELECT
        d.order_id,
        di.product_id,
        SUM(di.delivered_quantity) AS total_delivered
    FROM deliveries_clean d
    JOIN delivery_items_clean di
        ON d.delivery_id = di.delivery_id
    GROUP BY d.order_id, di.product_id
) delivered
    ON o.order_id = delivered.order_id
    AND oi.product_id = delivered.product_id

WHERE o.order_status = 'Delivered'

GROUP BY o.order_id
ORDER BY o.order_id;
"""

quantity_comparison = pd.read_sql_query(query, connection)

display(quantity_comparison.head(10))

,order_id,total_ordered_quantity,total_delivered_quantity
0,ORD0001,15,15
1,ORD0002,19,19
2,ORD0003,22,22
3,ORD0004,50,50
4,ORD0005,26,26
5,ORD0006,35,32
6,ORD0007,15,15
7,ORD0009,6,6
8,ORD0010,12,12
9,ORD0011,41,37


In [35]:
query = """
WITH product_quantities AS (
    SELECT
        o.order_id,
        oi.product_id,
        oi.ordered_quantity,
        COALESCE(d.delivered_quantity, 0) AS delivered_quantity
    FROM orders_clean o

    JOIN order_items_clean oi
        ON o.order_id = oi.order_id

    LEFT JOIN (
        SELECT
            d.order_id,
            di.product_id,
            SUM(di.delivered_quantity) AS delivered_quantity
        FROM deliveries_clean d
        JOIN delivery_items_clean di
            ON d.delivery_id = di.delivery_id
        GROUP BY d.order_id, di.product_id
    ) d
        ON o.order_id = d.order_id
        AND oi.product_id = d.product_id

    WHERE o.order_status = 'Delivered'
)

SELECT
    order_id,
    SUM(ordered_quantity) AS total_ordered,
    SUM(delivered_quantity) AS total_delivered,
    CASE
        WHEN SUM(
            CASE
                WHEN ordered_quantity = delivered_quantity THEN 0
                ELSE 1
            END
        ) = 0 THEN 1
        ELSE 0
    END AS in_full
FROM product_quantities
GROUP BY order_id
ORDER BY order_id;
"""

in_full_summary = pd.read_sql_query(query, connection)

display(in_full_summary.head(10))

,order_id,total_ordered,total_delivered,in_full
0,ORD0001,15,15,1
1,ORD0002,19,19,1
2,ORD0003,22,22,1
3,ORD0004,50,50,1
4,ORD0005,26,26,1
5,ORD0006,35,32,0
6,ORD0007,15,15,1
7,ORD0009,6,6,1
8,ORD0010,12,12,1
9,ORD0011,41,37,0


In [36]:
query = """
SELECT
    o.order_id,
    o.promised_date,
    MAX(d.actual_delivery_date) AS final_delivery_date,

    CASE
        WHEN MAX(d.actual_delivery_date) <= o.promised_date
        THEN 1
        ELSE 0
    END AS on_time

FROM orders_clean o

JOIN deliveries_clean d
    ON o.order_id = d.order_id

WHERE o.order_status = 'Delivered'

GROUP BY
    o.order_id,
    o.promised_date

ORDER BY o.order_id;
"""

on_time_summary = pd.read_sql_query(query, connection)

display(on_time_summary.head(10))

,order_id,promised_date,final_delivery_date,on_time
0,ORD0001,2026-03-05,2026-03-09,0
1,ORD0002,2026-01-19,2026-01-21,0
2,ORD0003,2026-05-24,2026-05-24,1
3,ORD0004,2026-05-11,2026-05-11,1
4,ORD0005,2026-05-02,2026-05-03,0
5,ORD0006,2026-03-16,2026-03-16,1
6,ORD0007,2026-02-27,2026-03-04,0
7,ORD0009,2026-02-20,2026-02-23,0
8,ORD0010,2026-08-07,2026-08-11,0
9,ORD0011,2026-01-19,2026-01-19,1


In [37]:
query = """
WITH product_quantities AS (
    SELECT
        o.order_id,
        oi.product_id,
        oi.ordered_quantity,
        COALESCE(d.delivered_quantity, 0) AS delivered_quantity
    FROM orders_clean o

    JOIN order_items_clean oi
        ON o.order_id = oi.order_id

    LEFT JOIN (
        SELECT
            d.order_id,
            di.product_id,
            SUM(di.delivered_quantity) AS delivered_quantity
        FROM deliveries_clean d
        JOIN delivery_items_clean di
            ON d.delivery_id = di.delivery_id
        GROUP BY d.order_id, di.product_id
    ) d
        ON o.order_id = d.order_id
        AND oi.product_id = d.product_id

    WHERE o.order_status = 'Delivered'
),

in_full AS (
    SELECT
        order_id,
        CASE
            WHEN SUM(
                CASE
                    WHEN ordered_quantity != delivered_quantity THEN 1
                    ELSE 0
                END
            ) = 0 THEN 1
            ELSE 0
        END AS in_full
    FROM product_quantities
    GROUP BY order_id
),

on_time AS (
    SELECT
        o.order_id,
        o.promised_date,
        MAX(d.actual_delivery_date) AS final_delivery_date,
        CASE
            WHEN MAX(d.actual_delivery_date) <= o.promised_date
            THEN 1
            ELSE 0
        END AS on_time
    FROM orders_clean o
    JOIN deliveries_clean d
        ON o.order_id = d.order_id
    WHERE o.order_status = 'Delivered'
    GROUP BY o.order_id, o.promised_date
)

SELECT
    t.order_id,
    t.promised_date,
    t.final_delivery_date,
    t.on_time,
    f.in_full,
    CASE
        WHEN t.on_time = 1 AND f.in_full = 1 THEN 1
        ELSE 0
    END AS otif
FROM on_time t
JOIN in_full f
    ON t.order_id = f.order_id
ORDER BY t.order_id;
"""

otif_results = pd.read_sql_query(query, connection)

display(otif_results.head(15))

,order_id,promised_date,final_delivery_date,on_time,in_full,otif
0,ORD0001,2026-03-05,2026-03-09,0,1,0
1,ORD0002,2026-01-19,2026-01-21,0,1,0
2,ORD0003,2026-05-24,2026-05-24,1,1,1
3,ORD0004,2026-05-11,2026-05-11,1,1,1
4,ORD0005,2026-05-02,2026-05-03,0,1,0
5,ORD0006,2026-03-16,2026-03-16,1,0,0
6,ORD0007,2026-02-27,2026-03-04,0,1,0
7,ORD0009,2026-02-20,2026-02-23,0,1,0
8,ORD0010,2026-08-07,2026-08-11,0,1,0
9,ORD0011,2026-01-19,2026-01-19,1,0,0


In [38]:
otif_summary = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_orders,
    SUM(otif) AS successful_otif_orders,
    COUNT(*) - SUM(otif) AS failed_otif_orders,
    ROUND(100.0 * SUM(otif) / COUNT(*), 2) AS otif_percentage
FROM (
    SELECT
        t.order_id,
        CASE
            WHEN t.on_time = 1 AND f.in_full = 1 THEN 1
            ELSE 0
        END AS otif
    FROM (
        SELECT
            o.order_id,
            CASE
                WHEN MAX(d.actual_delivery_date) <= o.promised_date
                THEN 1 ELSE 0
            END AS on_time
        FROM orders_clean o
        JOIN deliveries_clean d
            ON o.order_id = d.order_id
        WHERE o.order_status = 'Delivered'
        GROUP BY o.order_id, o.promised_date
    ) t

    JOIN (
        SELECT
            pq.order_id,
            CASE
                WHEN SUM(
                    CASE
                        WHEN pq.ordered_quantity != pq.delivered_quantity
                        THEN 1 ELSE 0
                    END
                ) = 0 THEN 1
                ELSE 0
            END AS in_full
        FROM (
            SELECT
                o.order_id,
                oi.product_id,
                oi.ordered_quantity,
                COALESCE(SUM(di.delivered_quantity), 0) AS delivered_quantity
            FROM orders_clean o
            JOIN order_items_clean oi
                ON o.order_id = oi.order_id
            LEFT JOIN deliveries_clean d
                ON o.order_id = d.order_id
            LEFT JOIN delivery_items_clean di
                ON d.delivery_id = di.delivery_id
                AND oi.product_id = di.product_id
            WHERE o.order_status = 'Delivered'
            GROUP BY o.order_id, oi.product_id, oi.ordered_quantity
        ) pq
        GROUP BY pq.order_id
    ) f
        ON t.order_id = f.order_id
);
""", connection)

display(otif_summary)

,total_orders,successful_otif_orders,failed_otif_orders,otif_percentage
0,455,122,333,26.81


In [39]:
query = """
WITH product_quantities AS (
    SELECT
        o.order_id,
        oi.product_id,
        oi.ordered_quantity,
        COALESCE(SUM(di.delivered_quantity), 0) AS delivered_quantity
    FROM orders_clean o
    JOIN order_items_clean oi
        ON o.order_id = oi.order_id
    LEFT JOIN deliveries_clean d
        ON o.order_id = d.order_id
    LEFT JOIN delivery_items_clean di
        ON d.delivery_id = di.delivery_id
        AND oi.product_id = di.product_id
    WHERE o.order_status = 'Delivered'
    GROUP BY o.order_id, oi.product_id, oi.ordered_quantity
),

in_full AS (
    SELECT
        order_id,
        CASE
            WHEN SUM(
                CASE WHEN ordered_quantity != delivered_quantity
                THEN 1 ELSE 0 END
            ) = 0 THEN 1
            ELSE 0
        END AS in_full
    FROM product_quantities
    GROUP BY order_id
),

on_time AS (
    SELECT
        o.order_id,
        CASE
            WHEN MAX(d.actual_delivery_date) <= o.promised_date
            THEN 1 ELSE 0
        END AS on_time
    FROM orders_clean o
    JOIN deliveries_clean d
        ON o.order_id = d.order_id
    WHERE o.order_status = 'Delivered'
    GROUP BY o.order_id, o.promised_date
)

SELECT
    COUNT(*) AS total_orders,
    SUM(t.on_time) AS on_time_orders,
    ROUND(100.0 * SUM(t.on_time) / COUNT(*), 2) AS on_time_percentage,
    SUM(f.in_full) AS in_full_orders,
    ROUND(100.0 * SUM(f.in_full) / COUNT(*), 2) AS in_full_percentage
FROM on_time t
JOIN in_full f
    ON t.order_id = f.order_id;
"""

performance_breakdown = pd.read_sql_query(query, connection)
display(performance_breakdown)

,total_orders,on_time_orders,on_time_percentage,in_full_orders,in_full_percentage
0,455,164,36.04,360,79.12


In [40]:
query = """
WITH product_quantities AS (
    SELECT
        o.order_id,
        oi.product_id,
        oi.ordered_quantity,
        COALESCE(SUM(di.delivered_quantity), 0) AS delivered_quantity
    FROM orders_clean o
    JOIN order_items_clean oi
        ON o.order_id = oi.order_id
    LEFT JOIN deliveries_clean d
        ON o.order_id = d.order_id
    LEFT JOIN delivery_items_clean di
        ON d.delivery_id = di.delivery_id
        AND oi.product_id = di.product_id
    WHERE o.order_status = 'Delivered'
    GROUP BY o.order_id, oi.product_id, oi.ordered_quantity
),

in_full AS (
    SELECT
        order_id,
        CASE WHEN SUM(
            CASE WHEN ordered_quantity != delivered_quantity
            THEN 1 ELSE 0 END
        ) = 0 THEN 1 ELSE 0 END AS in_full
    FROM product_quantities
    GROUP BY order_id
),

on_time AS (
    SELECT
        o.order_id,
        CASE WHEN MAX(d.actual_delivery_date) <= o.promised_date
        THEN 1 ELSE 0 END AS on_time
    FROM orders_clean o
    JOIN deliveries_clean d
        ON o.order_id = d.order_id
    WHERE o.order_status = 'Delivered'
    GROUP BY o.order_id, o.promised_date
),

combined AS (
    SELECT
        t.order_id,
        t.on_time,
        f.in_full
    FROM on_time t
    JOIN in_full f ON t.order_id = f.order_id
)

SELECT
    CASE
        WHEN on_time = 1 AND in_full = 1 THEN 'On Time and In Full'
        WHEN on_time = 0 AND in_full = 1 THEN 'Late but In Full'
        WHEN on_time = 1 AND in_full = 0 THEN 'On Time but Not In Full'
        ELSE 'Late and Not In Full'
    END AS performance_category,
    COUNT(*) AS total_orders,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM combined), 2)
        AS percentage
FROM combined
GROUP BY performance_category
ORDER BY total_orders DESC;
"""

failure_analysis = pd.read_sql_query(query, connection)
display(failure_analysis)

,performance_category,total_orders,percentage
0,Late but In Full,238,52.31
1,On Time and In Full,122,26.81
2,Late and Not In Full,53,11.65
3,On Time but Not In Full,42,9.23


In [41]:
query = """
CREATE VIEW IF NOT EXISTS otif_order_performance AS

WITH product_quantities AS (
    SELECT
        o.order_id,
        oi.product_id,
        oi.ordered_quantity,
        COALESCE(SUM(di.delivered_quantity), 0) AS delivered_quantity
    FROM orders_clean o
    JOIN order_items_clean oi
        ON o.order_id = oi.order_id
    LEFT JOIN deliveries_clean d
        ON o.order_id = d.order_id
    LEFT JOIN delivery_items_clean di
        ON d.delivery_id = di.delivery_id
        AND oi.product_id = di.product_id
    WHERE o.order_status = 'Delivered'
    GROUP BY o.order_id, oi.product_id, oi.ordered_quantity
),

in_full AS (
    SELECT
        order_id,
        CASE
            WHEN SUM(
                CASE WHEN ordered_quantity != delivered_quantity
                THEN 1 ELSE 0 END
            ) = 0 THEN 1
            ELSE 0
        END AS in_full
    FROM product_quantities
    GROUP BY order_id
),

on_time AS (
    SELECT
        o.order_id,
        o.promised_date,
        MAX(d.actual_delivery_date) AS final_delivery_date,
        CASE
            WHEN MAX(d.actual_delivery_date) <= o.promised_date
            THEN 1 ELSE 0
        END AS on_time
    FROM orders_clean o
    JOIN deliveries_clean d
        ON o.order_id = d.order_id
    WHERE o.order_status = 'Delivered'
    GROUP BY o.order_id, o.promised_date
)

SELECT
    t.order_id,
    t.promised_date,
    t.final_delivery_date,
    t.on_time,
    f.in_full,
    CASE
        WHEN t.on_time = 1 AND f.in_full = 1 THEN 1
        ELSE 0
    END AS otif
FROM on_time t
JOIN in_full f
    ON t.order_id = f.order_id;
"""

connection.execute(query)
connection.commit()

print("OTIF view created successfully!")

OTIF view created successfully!


In [42]:
display(pd.read_sql_query("""
SELECT *
FROM otif_order_performance
LIMIT 10;
""", connection))

,order_id,promised_date,final_delivery_date,on_time,in_full,otif
0,ORD0001,2026-03-05,2026-03-09,0,1,0
1,ORD0002,2026-01-19,2026-01-21,0,1,0
2,ORD0003,2026-05-24,2026-05-24,1,1,1
3,ORD0004,2026-05-11,2026-05-11,1,1,1
4,ORD0005,2026-05-02,2026-05-03,0,1,0
5,ORD0006,2026-03-16,2026-03-16,1,0,0
6,ORD0007,2026-02-27,2026-03-04,0,1,0
7,ORD0009,2026-02-20,2026-02-23,0,1,0
8,ORD0010,2026-08-07,2026-08-11,0,1,0
9,ORD0011,2026-01-19,2026-01-19,1,0,0


In [43]:
query = """
SELECT
    c.region,
    COUNT(*) AS total_orders,
    SUM(p.on_time) AS on_time_orders,
    SUM(p.in_full) AS in_full_orders,
    SUM(p.otif) AS successful_otif_orders,
    ROUND(100.0 * SUM(p.otif) / COUNT(*), 2) AS otif_percentage
FROM otif_order_performance p
JOIN orders_clean o
    ON p.order_id = o.order_id
JOIN customers_clean c
    ON o.customer_id = c.customer_id
GROUP BY c.region
ORDER BY otif_percentage DESC;
"""

region_analysis = pd.read_sql_query(query, connection)

display(region_analysis)

,region,total_orders,on_time_orders,in_full_orders,successful_otif_orders,otif_percentage
0,Unknown,9,5,9,5,55.56
1,East,91,35,78,29,31.87
2,West,144,54,116,39,27.08
3,South,93,30,69,23,24.73
4,North,118,40,88,26,22.03


In [44]:
query = """
SELECT
    c.customer_type,
    COUNT(*) AS total_orders,
    SUM(p.otif) AS successful_otif_orders,
    ROUND(100.0 * SUM(p.otif) / COUNT(*), 2) AS otif_percentage
FROM otif_order_performance p
JOIN orders_clean o
    ON p.order_id = o.order_id
JOIN customers_clean c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_type
ORDER BY otif_percentage DESC;
"""

customer_type_analysis = pd.read_sql_query(query, connection)

display(customer_type_analysis)

,customer_type,total_orders,successful_otif_orders,otif_percentage
0,Wholesale,209,60,28.71
1,Retail,96,27,28.13
2,Corporate,150,35,23.33


In [45]:
query = """
SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(oi.ordered_quantity) AS total_ordered,
    COALESCE(SUM(delivered.total_delivered), 0) AS total_delivered,
    SUM(oi.ordered_quantity)
        - COALESCE(SUM(delivered.total_delivered), 0) AS shortage_quantity
FROM order_items_clean oi

JOIN orders_clean o
    ON oi.order_id = o.order_id

JOIN products_clean p
    ON oi.product_id = p.product_id

LEFT JOIN (
    SELECT
        d.order_id,
        di.product_id,
        SUM(di.delivered_quantity) AS total_delivered
    FROM deliveries_clean d
    JOIN delivery_items_clean di
        ON d.delivery_id = di.delivery_id
    GROUP BY d.order_id, di.product_id
) delivered
    ON oi.order_id = delivered.order_id
    AND oi.product_id = delivered.product_id

WHERE o.order_status = 'Delivered'

GROUP BY p.product_id, p.product_name, p.category
ORDER BY shortage_quantity DESC;
"""

product_analysis = pd.read_sql_query(query, connection)

display(product_analysis)

,product_id,product_name,category,total_ordered,total_delivered,shortage_quantity
0,P004,Mouse,Accessories,890,862,28
1,P003,Keyboard,Uncategorized,990,964,26
2,P008,Webcam,Accessories,1206,1180,26
3,P011,USB Hub,Accessories,1164,1139,25
4,P007,Headset,Accessories,1057,1034,23
5,P009,Router,Networking,1027,1008,19
6,P006,Tablet,Computers,1117,1099,18
7,P001,Laptop,Computers,918,901,17
8,P010,Hard Drive,Storage,1132,1116,16
9,P002,Monitor,Accessories,948,936,12


In [46]:
query = """
SELECT
    d.warehouse_id,
    COUNT(DISTINCT d.order_id) AS total_orders,
    COUNT(DISTINCT CASE
        WHEN p.on_time = 1 THEN d.order_id
    END) AS on_time_orders,
    COUNT(DISTINCT CASE
        WHEN p.on_time = 0 THEN d.order_id
    END) AS late_orders,
    ROUND(
        100.0 * COUNT(DISTINCT CASE
            WHEN p.on_time = 1 THEN d.order_id
        END) / COUNT(DISTINCT d.order_id),
        2
    ) AS on_time_percentage
FROM deliveries_clean d

JOIN otif_order_performance p
    ON d.order_id = p.order_id

GROUP BY d.warehouse_id
ORDER BY on_time_percentage DESC;
"""

warehouse_analysis = pd.read_sql_query(query, connection)

display(warehouse_analysis)

,warehouse_id,total_orders,on_time_orders,late_orders,on_time_percentage
0,WH04,132,49,83,37.12
1,WH01,124,44,80,35.48
2,WH03,134,46,88,34.33
3,WH02,123,32,91,26.02


In [47]:
query = """
SELECT
    strftime('%Y-%m', o.order_date) AS order_month,
    COUNT(*) AS total_orders,

    SUM(p.on_time) AS on_time_orders,
    SUM(p.in_full) AS in_full_orders,
    SUM(p.otif) AS successful_otif_orders,

    ROUND(
        100.0 * SUM(p.on_time) / COUNT(*), 2
    ) AS on_time_percentage,

    ROUND(
        100.0 * SUM(p.in_full) / COUNT(*), 2
    ) AS in_full_percentage,

    ROUND(
        100.0 * SUM(p.otif) / COUNT(*), 2
    ) AS otif_percentage

FROM otif_order_performance p
JOIN orders_clean o
    ON p.order_id = o.order_id

GROUP BY strftime('%Y-%m', o.order_date)
ORDER BY order_month;
"""

monthly_analysis = pd.read_sql_query(query, connection)

display(monthly_analysis)

,order_month,total_orders,on_time_orders,in_full_orders,successful_otif_orders,on_time_percentage,in_full_percentage,otif_percentage
0,2026-01,45,15,32,10,33.33,71.11,22.22
1,2026-02,52,22,40,16,42.31,76.92,30.77
2,2026-03,44,15,38,12,34.09,86.36,27.27
3,2026-04,48,14,42,12,29.17,87.50,25.00
4,2026-05,54,20,47,19,37.04,87.04,35.19
5,2026-06,39,17,26,11,43.59,66.67,28.21
6,2026-07,45,21,33,13,46.67,73.33,28.89
7,2026-08,45,15,36,11,33.33,80.00,24.44
8,2026-09,33,14,26,11,42.42,78.79,33.33
9,2026-10,50,11,40,7,22.00,80.00,14.00


In [48]:
import os

os.makedirs("exports", exist_ok=True)

query = """
SELECT
    p.order_id,
    o.customer_id,
    c.customer_name,
    c.region,
    c.customer_type,
    o.order_date,
    p.promised_date,
    p.final_delivery_date,
    p.on_time,
    p.in_full,
    p.otif

FROM otif_order_performance p

JOIN orders_clean o
    ON p.order_id = o.order_id

JOIN customers_clean c
    ON o.customer_id = c.customer_id

ORDER BY p.order_id;
"""

order_performance = pd.read_sql_query(query, connection)

display(order_performance.head())

print("Total records:", len(order_performance))

,order_id,customer_id,customer_name,region,customer_type,order_date,promised_date,final_delivery_date,on_time,in_full,otif
0,ORD0001,C004,Customer 004,Unknown,Corporate,2026-02-27,2026-03-05,2026-03-09,0,1,0
1,ORD0002,C029,Customer 029,West,Corporate,2026-01-13,2026-01-19,2026-01-21,0,1,0
2,ORD0003,C018,Customer 018,West,Corporate,2026-05-21,2026-05-24,2026-05-24,1,1,1
3,ORD0004,C026,Customer 026,South,Retail,2026-05-06,2026-05-11,2026-05-11,1,1,1
4,ORD0005,C034,Customer 034,South,Retail,2026-04-25,2026-05-02,2026-05-03,0,1,0


Total records: 455


In [49]:
order_performance.to_csv(
    "exports/otif_order_performance.csv",
    index=False
)

print("Main dataset exported successfully!")

Main dataset exported successfully!


In [50]:
region_analysis.to_csv("exports/region_analysis.csv", index=False)

customer_type_analysis.to_csv("exports/customer_type_analysis.csv", index=False)

product_analysis.to_csv("exports/product_analysis.csv", index=False)

warehouse_analysis.to_csv("exports/warehouse_analysis.csv", index=False)

monthly_analysis.to_csv("exports/monthly_analysis.csv", index=False)

print("All analysis files exported successfully!")

All analysis files exported successfully!


In [51]:
import shutil

shutil.make_archive("otif_project_exports", "zip", "exports")

from google.colab import files
files.download("otif_project_exports.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>